# 04 — PaddleOCR: Baseline, Fine-tune, Ekspor
**Jalankan di sesi Colab baru.** Notebook ini tidak memakai PyTorch/Ultralytics, untuk menghindari bentrok library CUDA dengan PaddlePaddle.

Alur:
1. Pasang PaddlePaddle GPU 3.2.0 dan PaddleOCR 3.5 (sumber, untuk training).
2. **Baseline**: PP-OCRv5 mobile rec bawaan, tanpa fine-tune, pada data test.
3. **Fine-tune** PP-OCRv5 mobile rec dengan `train_list.txt` dan `val_list.txt`.
4. Evaluasi fine-tune pada test, lalu bandingkan dengan baseline.
5. Ekspor ke model inferensi, lalu ke **ONNX** untuk dijalankan OpenVINO di mini PC.

**Kamus karakter:** tetap memakai kamus bawaan PP-OCRv5 (sudah berisi A–Z dan 0–9). Dengan begitu bobot pretrained bagian *head* tetap terpakai penuh.
Pembatasan ke format plat dilakukan di pasca-proses (`fix_plate`).

**Runtime:** T4 GPU.

## 1. GPU dan Drive

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/0-KEPSTON/model_alpr'
OCR = f'{ROOT}/data/ocr'

## 2. Instalasi PaddlePaddle GPU + PaddleOCR
Perintah mengikuti dokumentasi resmi PaddleOCR (PaddlePaddle 3.2.0 untuk CUDA 12.6, driver ≥ 550.54.14).
Kalau `nvidia-smi` di sel 1 menunjukkan driver lebih lama, ganti `cu126` dengan `cu118`.

In [ ]:
!python -m pip install -q paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!python -m pip install -q "paddleocr>=3.0"
!git clone -q --depth 1 --branch release/3.5 https://github.com/PaddlePaddle/PaddleOCR.git /content/PaddleOCR
!python -m pip install -q -r /content/PaddleOCR/requirements.txt
import paddle
print('Paddle', paddle.__version__, '| GPU:', paddle.device.is_compiled_with_cuda())
paddle.utils.run_check()

In [ ]:
from pathlib import Path
import sys
CODE = Path(f'{ROOT}/code'); CODE.mkdir(parents=True, exist_ok=True)
(CODE / 'plate_utils.py').write_text('"""Utilitas plat nomor Indonesia: pembersihan, koreksi karakter per posisi, validasi pola."""\nimport re\n\nLETTER_FIX = {\'0\': \'O\', \'1\': \'I\', \'2\': \'Z\', \'4\': \'A\', \'5\': \'S\', \'6\': \'G\', \'7\': \'T\', \'8\': \'B\'}\nDIGIT_FIX = {\'O\': \'0\', \'D\': \'0\', \'Q\': \'0\', \'U\': \'0\', \'I\': \'1\', \'L\': \'1\', \'J\': \'1\',\n             \'Z\': \'2\', \'A\': \'4\', \'S\': \'5\', \'G\': \'6\', \'T\': \'7\', \'B\': \'8\'}\nPATTERN = re.compile(r\'^([A-Z]{1,2})([0-9]{1,4})([A-Z]{0,3})$\')\n\n# Kode wilayah yang umum dipakai. Dipakai sebagai penalti lunak (bukan penolakan),\n# sehingga kode yang tidak ada di daftar tetap bisa diterima bila tidak ada pilihan lain.\nREGION_CODES = set(\'A B D E F G H K L M N P R S T W Z\'.split()) | set(\n    \'AA AB AD AE AG BA BB BD BE BG BH BK BL BM BN BP DA DB DC DD DE DG DH DK DL DM DN DR DS DT \'\n    \'EA EB ED KB KH KT KU PA PB\'.split())\n\n\ndef clean(text):\n    """Huruf kapital, buang semua selain A-Z dan 0-9."""\n    return re.sub(r\'[^A-Z0-9]\', \'\', str(text).upper())\n\n\ndef fix_plate(text):\n    """Cari pembagian [huruf 1-2][angka 1-4][huruf 0-3] dengan koreksi karakter paling sedikit.\n    Mengembalikan string tanpa spasi (mis. \'B1234XYZ\') atau None bila tidak bisa dibentuk."""\n    s = clean(text)\n    if len(s) > 9 and s[-4:].isdigit():        # sisa baris masa berlaku (MM.YY) ikut terbaca\n        s = s[:-4]\n    best = None\n    for a in (1, 2):\n        for b in range(1, 5):\n            c = len(s) - a - b\n            if c < 0 or c > 3:\n                continue\n            out, cost, ok = [], 0, True\n            for i, ch in enumerate(s):\n                if i < a or i >= a + b:          # posisi huruf\n                    if ch.isalpha():\n                        out.append(ch)\n                    elif ch in LETTER_FIX:\n                        out.append(LETTER_FIX[ch]); cost += 1\n                    else:\n                        ok = False; break\n                else:                             # posisi angka\n                    if ch.isdigit():\n                        out.append(ch)\n                    elif ch in DIGIT_FIX:\n                        out.append(DIGIT_FIX[ch]); cost += 1\n                    else:\n                        ok = False; break\n            if ok:\n                if \'\'.join(out[:a]) not in REGION_CODES:\n                    cost += 2                     # kode wilayah tidak dikenal\n                key = (cost, -b)\n                if best is None or key < best[0]:\n                    best = (key, \'\'.join(out))\n    return best[1] if best else None\n\n\ndef is_valid(plate):\n    return bool(plate) and bool(PATTERN.match(plate))\n\n\ndef format_plate(plate):\n    """\'B1234XYZ\' -> \'B 1234 XYZ\'"""\n    m = PATTERN.match(plate or \'\')\n    return \' \'.join(g for g in m.groups() if g) if m else (plate or \'\')\n\n\ndef levenshtein(a, b):\n    a, b = a or \'\', b or \'\'\n    prev = list(range(len(b) + 1))\n    for i, ca in enumerate(a, 1):\n        cur = [i]\n        for j, cb in enumerate(b, 1):\n            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))\n        prev = cur\n    return prev[-1]\n')
sys.path.insert(0, str(CODE))
import importlib, plate_utils; importlib.reload(plate_utils)
from plate_utils import clean, fix_plate, is_valid, format_plate, levenshtein
print(fix_plate('b 1234 xyz'), fix_plate('B1Z34XY2'), fix_plate('BI234XYZ 05.28'))

## 3. Salin data OCR ke disk lokal

In [ ]:
import shutil, os
LOCAL = '/content/ocr_data'
if not os.path.exists(LOCAL):
    shutil.copytree(OCR, LOCAL, ignore=shutil.ignore_patterns('labels.csv'))
for f in ['train_list.txt', 'val_list.txt', 'test_list.txt', 'test_lokal_list.txt']:
    p = Path(LOCAL) / f
    print(f'{f:22s}', sum(1 for _ in open(p)) if p.exists() else '-', 'label')

## 4. Fungsi evaluasi
- **Exact match**: seluruh teks plat benar (setelah `fix_plate`). Ini metrik utama untuk laporan.
- **Akurasi karakter**: 1 − (jarak Levenshtein / panjang label).

In [ ]:
import pandas as pd

def load_list(name):
    p = Path(LOCAL) / name
    if not p.exists():
        return []
    return [ln.rstrip('\n').split('\t') for ln in open(p) if ln.strip()]

def get_text(r):
    try:
        return r['rec_text'], float(r['rec_score'])
    except Exception:
        j = r.json; j = j.get('res', j)
        return j['rec_text'], float(j['rec_score'])

def evaluate(rec_model, list_name, tag):
    items = load_list(list_name)
    if not items:
        print('Tidak ada data', list_name); return None
    rows = []
    for rel, gt in items:
        out = list(rec_model.predict(input=str(Path(LOCAL) / rel), batch_size=1))[0]
        raw, score = get_text(out)
        pred = fix_plate(raw) or clean(raw)
        rows.append({'crop': rel, 'gt': gt, 'raw': raw, 'pred': pred, 'score': score,
                     'exact': pred == gt, 'char_err': levenshtein(pred, gt)})
    res = pd.DataFrame(rows)
    exact = res['exact'].mean()
    char_acc = 1 - res['char_err'].sum() / res['gt'].str.len().sum()
    print(f'[{tag}] {list_name}: exact match = {exact:.1%} | akurasi karakter = {char_acc:.1%} | n = {len(res)}')
    res.to_csv(f'{ROOT}/eval/ocr_{tag}_{list_name.replace(".txt", "")}.csv', index=False)
    return res

## 5. Baseline: PP-OCRv5 mobile rec tanpa fine-tune

In [ ]:
from paddleocr import TextRecognition
os.makedirs(f'{ROOT}/eval', exist_ok=True)
base = TextRecognition(model_name='PP-OCRv5_mobile_rec')
base_res = {n: evaluate(base, n, 'baseline') for n in ['test_list.txt', 'test_lokal_list.txt']}

## 6. Fine-tune PP-OCRv5 mobile rec
Override dari konfigurasi bawaan:
- `epoch_num=100`, learning rate 0,0003 (lebih kecil dari default karena fine-tune), batch 64.
- Evaluasi setiap 200 iterasi; model terbaik disimpan sebagai `best_accuracy.pdparams` di Drive.
- `distributed=False` karena hanya 1 GPU.

Perkiraan waktu: ±30–90 menit, tergantung jumlah label.

In [ ]:
!wget -q -nc -P /content https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_mobile_rec_pretrained.pdparams
SAVE = f'{ROOT}/models/ppocr_rec_runs/PP-OCRv5_mobile_rec_plate'
CFG = 'configs/rec/PP-OCRv5/PP-OCRv5_mobile_rec.yml'
os.makedirs(SAVE, exist_ok=True)

In [ ]:
%cd /content/PaddleOCR
!python tools/train.py -c {CFG} -o \
  Global.pretrained_model=/content/PP-OCRv5_mobile_rec_pretrained.pdparams \
  Global.save_model_dir="{SAVE}" \
  Global.epoch_num=100 \
  Global.eval_batch_step="[0,200]" \
  Global.print_batch_step=20 \
  Global.distributed=False \
  Optimizer.lr.learning_rate=0.0003 \
  Train.dataset.data_dir={LOCAL} \
  Train.dataset.label_file_list="['{LOCAL}/train_list.txt']" \
  Train.sampler.first_bs=64 \
  Train.loader.num_workers=2 \
  Eval.dataset.data_dir={LOCAL} \
  Eval.dataset.label_file_list="['{LOCAL}/val_list.txt']" \
  Eval.loader.num_workers=2

### Kalau sesi terputus: jalankan sel 1–3 dan sel `wget`, lalu sel ini untuk melanjutkan dari checkpoint terakhir

In [ ]:
%cd /content/PaddleOCR
!python tools/train.py -c {CFG} -o \
  Global.checkpoints="{SAVE}/latest" \
  Global.save_model_dir="{SAVE}" \
  Global.epoch_num=100 Global.eval_batch_step="[0,200]" Global.distributed=False \
  Optimizer.lr.learning_rate=0.0003 \
  Train.dataset.data_dir={LOCAL} Train.dataset.label_file_list="['{LOCAL}/train_list.txt']" \
  Train.sampler.first_bs=64 Train.loader.num_workers=2 \
  Eval.dataset.data_dir={LOCAL} Eval.dataset.label_file_list="['{LOCAL}/val_list.txt']" Eval.loader.num_workers=2

## 7. Ekspor model inferensi (hasil fine-tune terbaik)

In [ ]:
INFER = f'{ROOT}/models/ppocrv5_plate_infer'
%cd /content/PaddleOCR
!python tools/export_model.py -c {CFG} -o \
  Global.pretrained_model="{SAVE}/best_accuracy.pdparams" \
  Global.save_inference_dir="{INFER}"
shutil.copy('/content/PaddleOCR/ppocr/utils/dict/ppocrv5_dict.txt', INFER)
print(os.listdir(INFER))

## 8. Evaluasi hasil fine-tune dan bandingkan dengan baseline

In [ ]:
ft = TextRecognition(model_name='PP-OCRv5_mobile_rec', model_dir=INFER)
ft_res = {n: evaluate(ft, n, 'finetune') for n in ['test_list.txt', 'test_lokal_list.txt']}

rows = []
for n in ['test_list.txt', 'test_lokal_list.txt']:
    for tag, r in [('Baseline (tanpa fine-tune)', base_res.get(n)), ('Fine-tune', ft_res.get(n))]:
        if r is not None:
            rows.append({'Data': n, 'Model': tag, 'Exact match': f"{r['exact'].mean():.1%}",
                         'Akurasi karakter': f"{1 - r['char_err'].sum() / r['gt'].str.len().sum():.1%}", 'n': len(r)})
cmp = pd.DataFrame(rows); cmp.to_csv(f'{ROOT}/eval/ocr_perbandingan.csv', index=False)
cmp

### Contoh kesalahan fine-tune (untuk analisis dan penambahan data)

In [ ]:
r = ft_res['test_list.txt']
r[~r['exact']].head(20)

## 9. Konversi ke ONNX untuk mini PC
PaddlePaddle versi CPU membutuhkan instruksi **AVX**, yang kemungkinan besar tidak dimiliki Celeron N3350.
Karena itu model OCR dikonversi ke **ONNX** dan dijalankan dengan **OpenVINO** (notebook 05), tanpa PaddlePaddle.

In [ ]:
!paddlex --install paddle2onnx
ONNX_DIR = f'{ROOT}/models/ppocrv5_plate_onnx'
!paddlex --paddle2onnx --paddle_model_dir "{INFER}" --onnx_model_dir "{ONNX_DIR}" --opset_version 11
for f in ['inference.yml', 'ppocrv5_dict.txt']:
    if os.path.exists(f'{INFER}/{f}'):
        shutil.copy(f'{INFER}/{f}', ONNX_DIR)
print(os.listdir(ONNX_DIR))

---
### Checklist 04
- [ ] Tabel perbandingan baseline vs fine-tune tersimpan (`eval/ocr_perbandingan.csv`)
- [ ] Exact match fine-tune pada test **≥ 90%** (kalau belum: tambah/perbaiki label di notebook 03 lalu latih ulang)
- [ ] `models/ppocrv5_plate_infer/` dan `models/ppocrv5_plate_onnx/` (berisi file `.onnx`) ada di Drive

**Berikutnya:** `05_pipeline_evaluasi.ipynb` — sesi Colab baru, runtime **CPU** cukup.